# Divye FE + Groupby Aggregation Features

Extends `s6e2-divye-fe.ipynb` with **groupby(CAT)[NUM].agg()** cross-feature statistics.

Three types of new features, all computed on combined train+test (no target → no leakage):

1. **Group mean**: `groupby(thallium)[age].mean()` — average age of patients with this thallium value  
2. **Group std**: `groupby(thallium)[age].std()` — variability within the group  
3. **Within-group z-score**: `(age - group_mean) / group_std` — *where does this patient fall within their group?*
   This is the most novel: it captures "is this a high-max_hr patient relative to others with the same thallium value?"
   CatBoost cannot easily derive this from raw features.

Feature count: 8 cat × 5 num × 3 stats = **120 new features**  
Total with Divye FE base: 67 + 120 = 187 features

Baseline: catboost_divye_fe OOF=0.95566  LB=0.95381

In [1]:
import subprocess
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

X      = train[ALL_FEATURES]
y      = train['heart_disease'].values
X_test = test[ALL_FEATURES]

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f'Train: {X.shape}  Test: {X_test.shape}')

Train: (630000, 13)  Test: (270000, 13)


In [2]:
# --- Feature engineering functions ---

def compute_freq_features(train_df, test_df, cols):
    combined = pd.concat([train_df[cols], test_df[cols]])
    tr_freq, te_freq = {}, {}
    for col in cols:
        freq_map = combined[col].value_counts(normalize=True)
        tr_freq[f'{col}_freq'] = train_df[col].map(freq_map).fillna(0).values
        te_freq[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0).values
    return tr_freq, te_freq


def compute_oof_te(train_df, test_df, cols, y, skf):
    global_mean = y.mean()
    oof_te  = {f'{col}_te': np.zeros(len(train_df)) for col in cols}
    test_te = {f'{col}_te': np.zeros(len(test_df))  for col in cols}
    for col in cols:
        key, tr_vals, te_vals, fold_test = f'{col}_te', train_df[col].values, test_df[col].values, []
        for tr_idx, val_idx in skf.split(train_df, y):
            te_map = pd.DataFrame({'v': tr_vals[tr_idx], 't': y[tr_idx]}).groupby('v')['t'].mean()
            oof_te[key][val_idx] = pd.Series(tr_vals[val_idx]).map(te_map).fillna(global_mean).values
            fold_test.append(pd.Series(te_vals).map(te_map).fillna(global_mean).values)
        test_te[key] = np.mean(fold_test, axis=0)
    return oof_te, test_te


def make_interaction_features(te_dict, top_cols):
    return {f'{c1}_x_{c2}': te_dict[c1] * te_dict[c2]
            for c1, c2 in combinations(top_cols, 2)}


def compute_groupby_features(train_df, test_df, cat_cols, num_cols):
    """Compute group mean, std, and within-group z-score for each cat×num pair.
    Computed on combined train+test — no target involved, no leakage risk.
    Z-score captures where a patient falls *within their categorical group*.
    """
    combined = pd.concat([train_df[cat_cols + num_cols],
                          test_df[cat_cols  + num_cols]], ignore_index=True)
    n_train = len(train_df)

    tr_gb, te_gb = {}, {}
    for cat in cat_cols:
        for num in num_cols:
            grp_mean = combined.groupby(cat)[num].transform('mean')
            grp_std  = combined.groupby(cat)[num].transform('std').fillna(1.0)
            zscore   = (combined[num] - grp_mean) / grp_std

            for suffix, vals in [('grp_mean', grp_mean),
                                  ('grp_std',  grp_std),
                                  ('grp_z',    zscore)]:
                name = f'{cat}_{num}_{suffix}'
                tr_gb[name] = vals.iloc[:n_train].values
                te_gb[name] = vals.iloc[n_train:].values

    return tr_gb, te_gb


def build_augmented(base_df, *dicts):
    df = base_df.copy().reset_index(drop=True)
    for d in dicts:
        for name, vals in d.items():
            df[name] = vals
    return df


# Precompute OOF TE to select top-8 for interactions
print('Frequency encoding...')
tr_freq, te_freq = compute_freq_features(X, X_test, ALL_FEATURES)

print('OOF target encoding...')
oof_te, test_te = compute_oof_te(X, X_test, ALL_FEATURES, y, SKF)

corrs = {col: abs(np.corrcoef(vals, y)[0, 1]) for col, vals in oof_te.items()}
top8  = sorted(corrs, key=corrs.get, reverse=True)[:8]
print(f'Top-8: {top8}')

tr_inter = make_interaction_features(oof_te,  top8)
te_inter = make_interaction_features(test_te, top8)

# Groupby features (computed once on combined data — no target, no leakage)
print('\nGroupby(cat)[num].{mean, std, z-score} features...')
tr_gb, te_gb = compute_groupby_features(X, X_test, CAT_FEATURES, NUM_FEATURES)
print(f'Groupby features: {len(tr_gb)}  ({len(CAT_FEATURES)} cats × {len(NUM_FEATURES)} nums × 3 stats)')

# Show z-score correlations with target to validate signal
print('\nTop z-score feature correlations with target:')
z_corrs = {k: abs(np.corrcoef(v, y)[0,1]) for k, v in tr_gb.items() if 'grp_z' in k}
for k, c in sorted(z_corrs.items(), key=lambda x: -x[1])[:8]:
    print(f'  {k:45s}  {c:.4f}')

Frequency encoding...
OOF target encoding...
Top-8: ['thallium_te', 'chest_pain_type_te', 'max_hr_te', 'number_of_vessels_fluro_te', 'st_depression_te', 'exercise_angina_te', 'slope_of_st_te', 'sex_te']

Groupby(cat)[num].{mean, std, z-score} features...
Groupby features: 120  (8 cats × 5 nums × 3 stats)

Top z-score feature correlations with target:
  fbs_over_120_max_hr_grp_z                      0.4403
  fbs_over_120_st_depression_grp_z               0.4298
  ekg_results_max_hr_grp_z                       0.4167
  ekg_results_st_depression_grp_z                0.4033
  sex_max_hr_grp_z                               0.3836
  sex_st_depression_grp_z                        0.3714
  slope_of_st_max_hr_grp_z                       0.3585
  exercise_angina_max_hr_grp_z                   0.3498


In [3]:
CATBOOST_PARAMS = dict(
    iterations        = 2000,
    learning_rate     = 0.05,
    depth             = 6,
    task_type         = 'CPU',
    thread_count      = -1,
    cat_features      = CAT_FEATURES,
    random_seed       = 42,
    verbose           = 0,
)

global_mean = y.mean()
oof_preds   = np.zeros(len(y))
test_folds  = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    print(f'\n=== Fold {fold + 1}/5 ===')
    X_tr_base  = X.iloc[tr_idx].reset_index(drop=True)
    X_val_base = X.iloc[val_idx].reset_index(drop=True)
    y_tr, y_val = y[tr_idx], y[val_idx]

    # Freq encoding
    fold_tr_freq, fold_te_freq  = compute_freq_features(X_tr_base, X_test,    ALL_FEATURES)
    _,            fold_val_freq = compute_freq_features(X_tr_base, X_val_base, ALL_FEATURES)

    # Per-fold TE
    fold_tr_te, fold_val_te, fold_te_te = {}, {}, {}
    for col in ALL_FEATURES:
        key    = f'{col}_te'
        te_map = pd.DataFrame({'v': X_tr_base[col].values, 't': y_tr}).groupby('v')['t'].mean()
        fold_tr_te[key]  = X_tr_base[col].map(te_map).fillna(global_mean).values
        fold_val_te[key] = X_val_base[col].map(te_map).fillna(global_mean).values
        fold_te_te[key]  = X_test[col].map(te_map).fillna(global_mean).values

    # Numeric interactions
    fold_tr_inter  = make_interaction_features(fold_tr_te,  top8)
    fold_val_inter = make_interaction_features(fold_val_te, top8)
    fold_te_inter  = make_interaction_features(fold_te_te,  top8)

    # Groupby features: recompute per fold from training rows only
    # (val rows excluded from group stats — strictly clean)
    fold_tr_gb, fold_val_gb, fold_te_gb = {}, {}, {}
    X_tr_full = pd.concat([X_tr_base, X_test], ignore_index=True)  # train fold + test
    n_tr = len(X_tr_base)
    for cat in CAT_FEATURES:
        for num in NUM_FEATURES:
            # Compute group stats from training fold + test (no val, no target)
            grp_mean_map = X_tr_full.groupby(cat)[num].mean()
            grp_std_map  = X_tr_full.groupby(cat)[num].std().fillna(1.0)

            for df_base, out_dict in [(X_tr_base, fold_tr_gb),
                                      (X_val_base, fold_val_gb),
                                      (X_test,     fold_te_gb)]:
                gm = df_base[cat].map(grp_mean_map).fillna(X_tr_base[num].mean()).values
                gs = df_base[cat].map(grp_std_map).fillna(1.0).values
                gz = (df_base[num].values - gm) / gs
                name_prefix = f'{cat}_{num}'
                out_dict[f'{name_prefix}_grp_mean'] = gm
                out_dict[f'{name_prefix}_grp_std']  = gs
                out_dict[f'{name_prefix}_grp_z']    = gz

    X_tr_aug  = build_augmented(X_tr_base,  fold_tr_freq,  fold_tr_te,  fold_tr_inter,  fold_tr_gb)
    X_val_aug = build_augmented(X_val_base, fold_val_freq, fold_val_te, fold_val_inter, fold_val_gb)
    X_te_aug  = build_augmented(X_test,     fold_te_freq,  fold_te_te,  fold_te_inter,  fold_te_gb)

    m = CatBoostClassifier(**CATBOOST_PARAMS)
    m.fit(X_tr_aug, y_tr, eval_set=(X_val_aug, y_val), early_stopping_rounds=50)

    oof_preds[val_idx] = m.predict_proba(X_val_aug)[:, 1]
    test_folds[fold]   = m.predict_proba(X_te_aug)[:, 1]
    print(f'Fold {fold+1}  AUC={roc_auc_score(y_val, oof_preds[val_idx]):.5f}  best_iter={m.best_iteration_}')

cv_auc     = roc_auc_score(y, oof_preds)
test_preds = test_folds.mean(axis=0)

print(f'\nOOF AUC (divye_fe_groupby): {cv_auc:.5f}')
print(f'Baseline divye_fe:           0.95566')
print(f'Delta:                       {cv_auc - 0.95566:+.5f}')


=== Fold 1/5 ===


/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = 

Fold 1  AUC=0.95606  best_iter=478

=== Fold 2/5 ===


/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = 

Fold 2  AUC=0.95488  best_iter=473

=== Fold 3/5 ===


...

Fold 3  AUC=0.95575  best_iter=566

=== Fold 4/5 ===


/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = 

Fold 4  AUC=0.95535  best_iter=466

=== Fold 5/5 ===


/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = vals
/tmp/ipykernel_1287361/667910541.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[name] = 

Fold 5  AUC=0.95618  best_iter=572

OOF AUC (divye_fe_groupby): 0.95564
Baseline divye_fe:           0.95566
Delta:                       -0.00002


In [4]:
np.save('submissions/oof_divye_fe_groupby.npy', oof_preds)

label = 'catboost_divye_fe_groupby'
fname = f'submissions/{label}.csv'
sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)

desc = f'{label} | cv_auc={cv_auc:.4f}'
print(f'Saved: {fname}')
print(f'desc:  {desc}')
print(f'OOF AUC: {cv_auc:.5f}  (baseline: 0.95566  delta: {cv_auc - 0.95566:+.5f})')

Saved: submissions/catboost_divye_fe_groupby.csv
desc:  catboost_divye_fe_groupby | cv_auc=0.9556
OOF AUC: 0.95564  (baseline: 0.95566  delta: -0.00002)


In [5]:
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')

catboost_divye_fe_groupby: submitted
